# Processing of neuronal data

This notebook guides through the process of populating tables found in `neuro_data.static_images` following experiments with static images. These steps are necessary to be able to train NN on this data.

In [1]:
from neuro_data.static_images import data_schemas as data, stats, configs
from neuro_data.static_images.data_schemas import MEI_STATIC, meso, fuse, anatomy, stimulus, experiment
#from staticnet_experiments import models
from staticnet_analyses import multi_mei, closed_loop

import datajoint as dj
from time import time
import numpy as np
from itertools import compress

dj.config['display.limit'] = 30

Connecting eywalker@10.28.0.34:3306


/src/static-networks/staticnet_analyses/multi_mei.py:1757: UserWarning: Use of this table is deprecated. It is kept only for record keeping purpose
  warnings.warn('Use of this table is deprecated. It is kept only for record keeping purpose')


# Listing of MEI scans

All MEI related scans are listed in the variable `MEI_STATIC`. **Add new MEI scans into the variable `MEI_STATIC`**.

In [2]:
MEI_STATIC

['(animal_id=20505 AND session=2 AND scan_idx=24)',
 '(animal_id=20505 AND session=3 AND scan_idx=7)',
 '(animal_id=20505 AND session=5 AND scan_idx=26)',
 '(animal_id=20505 AND session=6 AND scan_idx=1)',
 '(animal_id=20505 AND session=7 AND scan_idx=23)',
 '(animal_id=20505 AND session=7 AND scan_idx=29)',
 '(animal_id=20457 AND session=5 AND scan_idx=9)',
 '(animal_id=20457 AND session=5 AND scan_idx=17)',
 '(animal_id=20457 AND session=5 AND scan_idx=27)',
 '(animal_id=20457 AND session=7 AND scan_idx=4)',
 '(animal_id=20457 AND session=7 AND scan_idx=10)',
 '(animal_id=20457 AND session=7 AND scan_idx=16)',
 '(animal_id=20457 AND session=8 AND scan_idx=9)',
 '(animal_id=20457 AND session=8 AND scan_idx=12)',
 '(animal_id=20457 AND session=8 AND scan_idx=22)',
 '(animal_id=20505 AND session=10 AND scan_idx=14)',
 '(animal_id=20505 AND session=10 AND scan_idx=19)',
 '(animal_id=20505 AND session=11 AND scan_idx=16)',
 '(animal_id=20505 AND session=12 AND scan_idx=16)',
 '(animal_id=

# Exclude bad trials

Find trials with improper number of flip times

In [3]:
targets = [{'animal_id': 20210, 'session':8, 'scan_idx': 17}]

In [4]:
fuse.ScanDone & targets

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,pipe_version,segmentation_method,spike_method spike inference method,pipe pipeline name
20210,8,17,1,6,5,meso


In [5]:
unprocessed = (fuse.ScanDone & targets) - data.InputResponse

In [6]:
unprocessed

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,pipe_version,segmentation_method,spike_method spike inference method,pipe pipeline name


In [7]:
keys, flips = (stimulus.Trial & stimulus.Frame & unprocessed - data.ExcludedTrial).fetch('KEY', 'flip_times')

In [8]:
n_flips = np.array([f.shape[1] for f in flips])

In [9]:
len(n_flips)

0

In [10]:
n_flips

array([], dtype=float64)

There should only be 3 frames for `stimulus.Frame` stimulus.

In [11]:
correct_flips = 3

In [12]:
bad_trials = (n_flips != correct_flips)
sum(bad_trials)

0

In [13]:
bad_trial_keys = list(compress(keys, bad_trials))

In [14]:
data.ExcludedTrial.insert(bad_trial_keys)

In [15]:
data.ExcludedTrial()

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,trial_idx trial index within sessions,exclusion_comment reasons for exclusion
20210,4,11,0,
20210,5,16,0,
20210,5,26,0,
20210,7,10,0,
20457,7,4,0,
20457,8,9,0,
20505,10,19,0,
20505,11,16,0,
20505,12,16,0,
20505,14,33,0,


# Processing of the data

**NOTE:** All of the following steps can be and should be completed by invoking `closed_loop.fill_data(cond)`. In particular, filling of `ConditionTier` and `InputResponse` should **not** be performed in a notebook due to long output.

Add the new scan information into `MEI_STATIC` variable defined in `data_schemas.py`

In [16]:
target_scan = {'animal_id': 20892, 'session': 9, 'scan_idx': 10}

In [17]:
%%time
data.StaticScan().populate(targets)

CPU times: user 18.8 ms, sys: 558 µs, total: 19.3 ms
Wall time: 37.2 ms


In [18]:
%%time
data.ConditionTier().populate(targets)

CPU times: user 28.6 ms, sys: 841 µs, total: 29.4 ms
Wall time: 53.7 ms


In [19]:
%%time
data.Frame.populate('preproc_id=0')

CPU times: user 26.6 ms, sys: 519 µs, total: 27.1 ms
Wall time: 2.14 s


In [20]:
%%time
data.InputResponse().populate(targets, 'preproc_id=0', suppress_errors=True)

CPU times: user 12.2 ms, sys: 7.25 ms, total: 19.4 ms
Wall time: 27.4 ms


[]

Populating `Eye()` requires that `pupil.FittedContour` is already filled (which requires that `pupil.ManuallyTrackedContours` is already filled).

In [21]:
%%time
data.Eye().populate(targets)

CPU times: user 196 ms, sys: 3.81 ms, total: 199 ms
Wall time: 766 ms


In [22]:
%%time
data.Treadmill().populate(targets)

CPU times: user 28 ms, sys: 8.26 ms, total: 36.2 ms
Wall time: 76.7 ms


## Fill Area and Layer map with V1 L2/3

Most MEI scans have been performed on V1 L2/3. Check if True.

In [23]:
missing_scans = (meso.ScanDone & targets) - (anatomy.AreaMembership * anatomy.LayerMembership)

In [24]:
missing_scans

animal_id id number,session session index for the mouse,scan_idx number of TIFF stack file,pipe_version,segmentation_method,spike_method spike inference method


In [25]:
units = (meso.ScanSet.Unit & missing_scans).fetch('KEY')

In [26]:
for u in units:
    u['brain_area'] = 'V1'
    u['layer'] = 'L2/3'

In [27]:
anatomy.AreaMembership.insert(units, ignore_extra_fields=True, allow_direct_insert=True, skip_duplicates=True)

In [28]:
anatomy.LayerMembership.insert(units, ignore_extra_fields=True, allow_direct_insert=True, skip_duplicates=True)

Update `StaticMultiDataset`'s `selection` list inside `fill` method to add new scans/preprocessing combination.

## Create Static Dataset
You'll need to add the new dataset in the selection variable of the fill function

In [29]:
data.StaticMultiDataset().fill()

Already found entry {'group_id': 0, 'description': '11521-7-1'}
Already found entry {'group_id': 1, 'description': '11521-7-2'}
Already found entry {'group_id': 2, 'description': '16157-5-5'}
Already found entry {'group_id': 3, 'description': '16157-5-6'}
Already found entry {'group_id': 4, 'description': '16157-5-5-scaled'}
Already found entry {'group_id': 5, 'description': '16312-3-20'}
Already found entry {'group_id': 6, 'description': '11521-7-1-scaled'}
Already found entry {'group_id': 7, 'description': '11521-7-2-scaled'}
Already found entry {'group_id': 8, 'description': '18765-4-6'}
Already found entry {'group_id': 9, 'description': '16157-5'}
Already found entry {'group_id': 10, 'description': '20505-2-24'}
Already found entry {'group_id': 11, 'description': '20505-3-7'}
Already found entry {'group_id': 12, 'description': '20505-6-1'}
Already found entry {'group_id': 13, 'description': '20505-7-29'}
Already found entry {'group_id': 14, 'description': '20457-5-9'}
Already found